# CoChem-SCRIBE: Autonomous Scientific Manuscript & Technical Report Dashboard

**Air-Gap Verified Voila Entry Point for Stage 6.0-6.3 Autonomous Documentation**

This interactive notebook orchestrates the SCRIBE documentation pipeline, providing real-time telemetry, LLM routing controls, and automated LaTeX/Markdown report compilation.

In [ ]:
import sys
import os
import json
from pathlib import Path
from IPython.display import display, HTML
from pydantic import BaseModel, ValidationError

# 1. Resolve workspace root and ensure repository modules are accessible
repo_root: Path = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

# Pydantic model for JSON validation
class CoChemSystemConfig(BaseModel, extra="allow"):
    pass

# 2. Stage 0.0 Environment Handshake & Air-Gap Verification
custom_artifacts_env: str | None = os.environ.get("COCHEM_ARTIFACTS_DIR") or os.environ.get("COCHEM_ARTIFACT_DIR")
if custom_artifacts_env:
    config_path: Path = Path(custom_artifacts_env) / "Registry" / "cochem_system_config.json"
else:
    config_path: Path = Path.home() / "CoChem_Artifacts" / "Registry" / "cochem_system_config.json"

is_environment_valid: bool = False

if not config_path.exists():
    display(HTML("<div style='color:red;'><b>CoChemError:</b> Cannot launch SCRIBE: Missing cochem_system_config.json. Please run CoChem-CORE Stage 0.0.</div>"))
else:
    try:
        with open(config_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        if not isinstance(raw_data, dict) or len(raw_data) == 0:
            display(HTML("<div style='color:red;'><b>CoChemError:</b> Cannot launch SCRIBE: Invalid cochem_system_config.json structure. Please run CoChem-CORE Stage 0.0.</div>"))
        else:
            try:
                config_data: CoChemSystemConfig = CoChemSystemConfig.model_validate(raw_data)
                is_environment_valid = True
            except ValidationError as val_error:
                display(HTML(f"<div style='color:red;'><b>CoChemError:</b> Cannot launch SCRIBE: Invalid cochem_system_config.json structure ({val_error}). Please run CoChem-CORE Stage 0.0.</div>"))
    except Exception as parse_error:
        display(HTML(f"<div style='color:red;'><b>CoChemError:</b> Cannot launch SCRIBE: Corrupt cochem_system_config.json ({parse_error}). Please run CoChem-CORE Stage 0.0.</div>"))

## Stage 6.0: Voila GUI Instantiation

Imports `ScribeDashboard` from `ui.voila_layout.scribe_gui_dashboard` and renders the interactive dashboard.

In [ ]:
if "is_environment_valid" in globals() and is_environment_valid:
    from ui.voila_layout.scribe_gui_dashboard import ScribeDashboard

    dashboard: ScribeDashboard = ScribeDashboard()
    dashboard.display()